In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import DebertaTokenizer
from tqdm import tqdm

In [ ]:
df = pd.read_csv('cleaned_movie_reviews.csv', engine='python')
texts = df['clean_review_content'].astype(str).tolist()
labels = df['label'].tolist()

In [ ]:
# Initialize DeBERTa Tokenizer
tokenizer = DebertaTokenizer.from_pretrained('microsoft/deberta-base')

# Batch size to avoid memory issues
batch_size = 10000
input_ids, attention_masks, all_labels = [], [], []

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

In [ ]:
# Tokenization in Chuncks

for i in tqdm(range(0, len(texts), batch_size), desc="Tokenizing"):
    batch_texts = texts[i:i+batch_size]
    batch_labels = labels[i:i+batch_size]

    encoding = tokenizer(
        batch_texts,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )

    input_ids.append(encoding['input_ids'])
    attention_masks.append(encoding['attention_mask'])
    all_labels.append(torch.tensor(batch_labels, dtype=torch.float32))

# Concatination of all the batches
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
labels = torch.cat(all_labels, dim=0)

Tokenizing: 100%|██████████| 106/106 [06:00<00:00,  3.40s/it]


In [ ]:
# Saving all the tokenized data
torch.save({
    'input_ids': input_ids,
    'attention_masks': attention_masks,
    'labels': labels,
}, 'deberta_tokenized_data.pt')

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
data = torch.load('deberta_tokenized_data.pt')

input_ids = data['input_ids']
attention_masks = data['attention_masks']
labels = data['labels']

In [ ]:
# precautious attempt to ensure that labels are converted to numpy

labels_np = labels.numpy() if isinstance(labels, torch.Tensor) else labels

# indexing labels

indicies = np.arange(len(labels_np))

In [ ]:
# 80% train split, 20% temporary split from entire dataset
train_idx, temp_idx = train_test_split(
    indicies, test_size=0.2, random_state=42, stratify=labels_np)

In [ ]:
# 50% (10% from the original dataset) Validation Split and 50% (10% from the original dataset) Test Split from Temporary Split
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, random_state=42, stratify=labels_np[temp_idx]
)

In [ ]:
# Creating Splits
train_inputs = input_ids[train_idx]
train_masks = attention_masks[train_idx]
train_labels = labels[train_idx]

val_inputs = input_ids[val_idx]
val_masks = attention_masks[val_idx]
val_labels = labels[val_idx]

test_inputs = input_ids[val_idx]
test_masks = attention_masks[val_idx]
test_labels = labels[val_idx]

In [ ]:
torch.save({
    'input_ids': train_inputs,
    'attention_masks': train_masks,
    'labels': train_labels,
}, 'train_data.pt')

torch.save({
    'input_ids': val_inputs,
    'attention_masks': val_masks,
    'labels': val_labels,
}, 'validation_data.pt')

torch.save({
    'input_ids': test_inputs,
    'attention_masks': test_masks,
    'labels': test_labels,
}, 'test_data.pt')

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
try:
  batch_size = 16

  train_dataset = TensorDataset(train_inputs, train_masks, train_labels)
  val_dataset = TensorDataset(val_inputs, val_masks, val_labels)
  test_dataset = TensorDataset(test_inputs, test_masks, test_labels)

  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  val_loader = DataLoader(val_dataset, batch_size=batch_size)
  test_loader = DataLoader(test_dataset, batch_size=batch_size)

  print(f"Batch size {batch_size} works!")

except RuntimeError as e:
    print(f"Batch size {batch_size} is too large: {e}")

Batch size 16 works!


In [ ]:
from transformers import DebertaForSequenceClassification

In [ ]:
model = DebertaForSequenceClassification.from_pretrained(
    'microsoft/deberta-base',
    num_labels=2
)

pytorch_model.bin:   0%|          | 0.00/559M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/559M [00:00<?, ?B/s]

Some weights of DebertaForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Fine-Tuning the Model

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
model.to(device)

DebertaForSequenceClassification(
  (deberta): DebertaModel(
    (embeddings): DebertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=0)
      (LayerNorm): DebertaLayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaLayer(
          (attention): DebertaAttention(
            (self): DisentangledSelfAttention(
              (in_proj): Linear(in_features=768, out_features=2304, bias=False)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (pos_proj): Linear(in_features=768, out_features=768, bias=False)
              (pos_q_proj): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): DebertaLayerNorm()
              (dropout): Dropout(p=0

In [ ]:
epochs = 3
optimizer = AdamW(model.parameters(), lr=2e-5)
total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=(0.1 * total_steps),
    num_training_steps=total_steps
)

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score

In [ ]:
for epoch in range(epochs):
  model.train()
  total_loss = 0

  for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}"):
    input_ids, attention_masks, labels = [b.to(device) for b in batch]

    optimizer.zero_grad()
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_masks,
        labels=labels.long()
    )

    loss = outputs.loss
    loss.backward()
    optimizer.step()
    scheduler.step()

    total_loss += loss.item()

  average_total_loss = total_loss / len(train_loader)
  print(f"Epoch {epoch+1} | Train Loss: {average_total_loss:.4f}")

  # Validation

  model.eval()
  preds, true = [], []
  with torch.no_grad():
    for batch in tqdm(val_loader, desc=f"Validation"):
      input_ids, attention_masks, labels = [b.to(device) for b in batch]

      outputs = model(
          input_ids=input_ids,
          attention_mask=attention_masks
      )

      logits = outputs.logits
      preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
      true.extend(labels.cpu().numpy())

  val_accuracy = accuracy_score(true, preds)
  val_f1 = f1_score(true, preds)
  print(f"Epoch {epoch+1} | Validation Accuracy: {val_accuracy:.4f} | Validation F1: {val_f1:.4f}")

Epoch 1:  22%|██▏       | 11370/52727 [1:21:11<4:54:22,  2.34it/s]

In [ ]:
model.eval()
test_preds, test_true = [], []
with torch.no_grad():
  for batch in test_loader:
    input_ids, attention_masks, labels = [b.to(device) for b in batch]
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_masks
    )
    logits = outputs.logits
    test_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
    test_true.extend(labels.cpu().numpy())

test_acc = accuracy_score(test_true, test_preds)
test_f1 = f1_score(test_true, test_preds)
print(f"Test Acc: {test_acc:.4f}, Test F1: {test_f1:.4f}")